# Distribution Shifts

Compare each stock's daily return distribution with the previous calendar month. This can reveal when a stock's returns change, including around the onset of the COVID-19 pandemic.

In [ ]:
%load_ext dotenv
%dotenv
%run update_path.py

from utils.logger import get_logger
_logs = get_logger(__name__)

In [ ]:
import dask
dask.config.set({'dataframe.query-planning': True})
import dask.dataframe as dd
import pandas as pd
import os
from glob import glob
from IPython.display import display

In [ ]:
ft_dir = os.getenv("FEATURES_DATA")
ft_glob = glob(ft_dir+'/*.parquet')
df = dd.read_parquet(ft_glob).compute().reset_index()

## Data Preparation

+ Group daily returns by stock and calendar month.
+ Compare only consecutive calendar months with at least two valid returns in each month. The result includes sample sizes so short months are visible.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M')
print(f"Available dates: {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d}")

In [ ]:
month_coverage = df.groupby('Month').agg(
    first_date=('Date', 'min'),
    last_date=('Date', 'max'),
    trading_days=('Date', 'nunique'),
)
display(month_coverage.tail(12).style.format({
    'first_date': '{:%Y-%m-%d}',
    'last_date': '{:%Y-%m-%d}',
}).set_caption('Dates available in the latest 12 months'))

# Kolmogorov-Smirnov Test

+ Use `scipy.stats.ks_2samp` to compare each stock's returns in consecutive months. The null hypothesis is that the two monthly distributions are the same.
+ The final report includes only comparisons with p < 0.01. Means and standard deviations use the same daily returns as the test.
+ These p-values are unadjusted for the many stock-month comparisons; some apparent changes may occur by chance.

In [ ]:
from scipy.stats import ks_2samp

ks_rows = []

for ticker, stock_data in df.groupby('ticker'):
    monthly_data = [
        (month, rows['Returns'].dropna())
        for month, rows in stock_data.groupby('Month')
    ]
    for (before_month, before), (after_month, after) in zip(monthly_data, monthly_data[1:]):
        if after_month != before_month + 1 or len(before) < 2 or len(after) < 2:
            continue

        result = ks_2samp(before, after)
        ks_rows.append({
            'Stock': ticker,
            'Before month': str(before_month),
            'After month': str(after_month),
            'Before n': len(before),
            'After n': len(after),
            'Before mean': before.mean(),
            'After mean': after.mean(),
            'Before SD': before.std(),
            'After SD': after.std(),
            'KS statistic': result.statistic,
            'p-value': result.pvalue,
        })

ks_results = pd.DataFrame(ks_rows)
significant_changes = (
    ks_results.loc[ks_results['p-value'] < 0.01]
    .sort_values(['After month', 'p-value', 'Stock'], ascending=[False, True, True])
    .reset_index(drop=True)
)
print(f'{len(significant_changes)} stock-month pairs with p < 0.01 among {len(ks_results)} comparisons')
display(
    significant_changes.style.format({
        'Before mean': '{:.2%}',
        'After mean': '{:.2%}',
        'Before SD': '{:.2%}',
        'After SD': '{:.2%}',
        'KS statistic': '{:.3f}',
        'p-value': '{:.3g}',
    }).set_caption('Flagged stock-month comparisons (p < 0.01), newest first')
)